# Script for downloading documents from tweede kamer API


In [58]:
# imports

# from ai_classifier import AIClassifier


import requests
import csv
import json
from pathlib import Path
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from bs4 import BeautifulSoup
import pandas as pd

import pdfplumber
from io import BytesIO
from urllib.parse import urljoin

## part 1: making json/csv with links to pdf's

In [59]:
# Configuration
BASE_URL = "https://opendata.rijksoverheid.nl/v1/documents"
OUTPUT_JSON = "kamerstukken_rijksoverheid.json"
OUTPUT_CSV = "kamerstukken_rijksoverheid.csv"

DOC_TYPE = "kamerstuk"   # <-- uit <name> in /v1/documents/infotypes
ROWS = 200                  # max per call (API-max)
MAX_RECORDS = 50000         # veiligheidslimiet


In [60]:
# HTTP sessie met retries
def make_session():
    sess = requests.Session()
    sess.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/605.1.15 (KHTML, like Gecko) "
            "Version/18.1.1 Safari/605.1.15"
        ),
        "Accept": "application/json",
        "Accept-Language": "nl-NL,nl;q=0.9,en;q=0.8",
    })
    retries = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    return sess


# Hoofdlogica: alle vergaderstukken ophalen

def fetch_vergaderstukken():
    session = make_session()
    offset = 0
    all_docs = []

    # --- Probeer total op te halen, maar crash niet als structuur anders is ---
    total_found = None
    try:
        test_params = {
            "type": DOC_TYPE,
            "output": "json",
            "rows": 1,
            "offset": 0,
        }
        test_resp = session.get(BASE_URL, params=test_params, timeout=30)
        test_json = test_resp.json()

        # Alleen gebruiken als het een dict is en 'total' bevat
        if isinstance(test_json, dict) and "total" in test_json:
            total_found = test_json["total"]
    except Exception as e:
        print(f"[WARN] Kon 'total' niet bepalen: {e}")

    # --- Setup tqdm ---
    if total_found:
        print(f"[INFO] API geeft totaal: {total_found}")
        total_batches = (total_found // ROWS) + 2
        pbar = tqdm(total=total_batches, desc="Ophalen batches")
    else:
        print("[INFO] Geen totaal beschikbaar — gebruik open-ended tqdm")
        pbar = tqdm(desc="Ophalen batches")

    # --- Ophalen batches ---
    while True:
        params = {
            "type": DOC_TYPE,
            "output": "json",
            "rows": ROWS,
            "offset": offset,
        }

        r = session.get(BASE_URL, params=params, timeout=30)
        if r.status_code != 200:
            print(f"[WARN] HTTP {r.status_code} bij offset={offset}, stop.")
            break

        try:
            data = r.json()
        except Exception as e:
            print(f"[WARN] JSON parse error bij offset={offset}: {e}")
            break

        docs = data.get("documents", []) if isinstance(data, dict) else data
        if not docs:
            print("[INFO] Geen documenten meer (lege batch), klaar.")
            break

        all_docs.extend(docs)
        offset += ROWS
        pbar.update(1)

        if offset >= MAX_RECORDS:
            print(f"[INFO] MAX_RECORDS ({MAX_RECORDS}) bereikt, stoppen.")
            break

    pbar.close()
    print(f"[INFO] Totaal opgehaalde vergaderstukken: {len(all_docs)}")
    return all_docs


# Opslaan

def save_json(records):
    path = Path(OUTPUT_JSON)
    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"[INFO] JSON opgeslagen in {path.resolve()}")

def save_csv(records):
    path = Path(OUTPUT_CSV)

    # kies een subset van velden die bijna altijd voorkomen
    fieldnames = [
        "id",
        "title",
        "introduction",
        "canonical",
        "dataurl",
        "frontenddate",
        "lastmodified",
        "available",
    ]

    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for d in records:
            row = {k: d.get(k) for k in fieldnames}
            w.writerow(row)

    print(f"[INFO] CSV opgeslagen in {path.resolve()}")



In [61]:
docs = fetch_vergaderstukken()
save_json(docs)
save_csv(docs)

[WARN] Kon 'total' niet bepalen: Expecting value: line 1 column 1 (char 0)
[INFO] Geen totaal beschikbaar — gebruik open-ended tqdm


Ophalen batches: 0it [00:00, ?it/s]

[WARN] HTTP 400 bij offset=0, stop.
[INFO] Totaal opgehaalde vergaderstukken: 0


[INFO] JSON opgeslagen in C:\Users\joly-\Github\HUMAN\tweede_kamer\kamerstukken_rijksoverheid.json
[INFO] CSV opgeslagen in C:\Users\joly-\Github\HUMAN\tweede_kamer\kamerstukken_rijksoverheid.csv


# cleaning df and optionally splitting

In [49]:
df = pd.read_csv("beleidsnotas_rijksoverheid.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23743 entries, 0 to 23742
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            23743 non-null  str  
 1   title         23743 non-null  str  
 2   introduction  23705 non-null  str  
 3   canonical     23743 non-null  str  
 4   dataurl       23743 non-null  str  
 5   frontenddate  23743 non-null  str  
 6   lastmodified  23743 non-null  str  
 7   available     23743 non-null  str  
dtypes: str(8)
memory usage: 1.4 MB


In [48]:


#change frontenddate to clear date format
df['frontenddate'] = pd.to_datetime(df['frontenddate'], errors='coerce').dt.date

# show distribution of years in frontenddate
df['year'] = pd.DatetimeIndex(df['frontenddate']).year

# #delete frontenddate year column
df2 = df.drop(columns=["frontenddate"])

# delete zero after the year in 'year' column
df['year'] = df['year'].fillna(0).astype(int)

# counts = df['year'].value_counts().sort_index()
# print(counts)

#only keep columns with year 2015-2025
df_filtered = df[(df['year'] >= 2015) & (df['year'] <= 2025)]

# #how many rows dropped
# rows_dropped = len(df) - len(df_filtered)
# print(f"Rijen verwijderd: {rows_dropped}")

df_filtered = df_filtered.drop(columns={"frontenddate", "available", "introduction"})
df_filtered.head()

#---------------optional: split if files is too large to run

# #split dataframe into chunks of 500 rows
# chunk_size = 500
# num_chunks = (len(df_filtered) // chunk_size) + 1
# for i in range(num_chunks):
#     chunk = df_filtered[i*chunk_size:(i+1)*chunk_size]
#     chunk.to_csv(f"beleidsnotas_subset_part_{i+1}.csv", index=False)
#     print(f"[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_{i+1}.csv")

,id,title,canonical,dataurl,lastmodified,year
6,7d555f4a-af1d-4fbe-b5ed-b3fd22c7f0da,Bijlage short term averages,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2025-03-11T22:32:04.128Z,2015
7,3d2b6181-20c9-44d8-b56b-25d6cfab5deb,Contourennota Modernisering Wetboek van Strafv...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2025-05-08T13:25:21.507Z,2016
8,6d0e2cd9-7f1d-4b15-abbf-c8d4c1bc32ce,Investeren in Perspectief (Beleidsnota 2018),https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2025-09-10T12:45:40.214Z,2018
9,5e787815-58d1-40a7-afc9-54c0720ffce4,"Visie Landbouw, Natuur en Voedsel: Waardevol e...",https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2025-09-10T12:46:21.796Z,2018
10,32171765-2533-4c46-b2e8-8ae0cf446860,Nota Defensie Industrie Strategie,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2025-09-10T12:49:25.652Z,2018


## part 2: downloading pdfs from links in csv

In [50]:
#finding PDF links

def find_pdf_url(canonical_url: str) -> str | None:
    """
    Haalt de canonical pagina op en probeert een PDF-link te vinden.

    Werkt voor o.a.:
    - klassieke .pdf-links
    - open.overheid.nl/documenten/.../file (zoals jouw voorbeeld)
    - open.overheid.nl/documenten/.../pdf
    - links waarvan de tekst 'PDF' bevat
    """
    if not canonical_url:
        return None

    try:
        resp = requests.get(canonical_url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] pagina niet bereikbaar: {canonical_url} ({e})")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    # 1) open.overheid.nl/documenten/.../file (meest betrouwbare voor veel docs)
    a = soup.select_one('a[href*="open.overheid.nl/documenten/"][href$="/file"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 2) open.overheid.nl/documenten/.../pdf
    a = soup.select_one('a[href*="open.overheid.nl/documenten/"][href$="/pdf"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 3) Klassieke .pdf-link ergens anders
    a = soup.select_one('a[href$=".pdf"], a[href*=".pdf"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 4) Fallback: elk <a> met 'pdf' in de link-tekst
    for link in soup.find_all("a"):
        text = (link.get_text() or "").strip().lower()
        href = link.get("href")
        if "pdf" in text and href:
            return urljoin(canonical_url, href)

    print(f"[WARN] geen pdf-link gevonden op: {canonical_url}")
    return None


In [51]:
# 2) PDF downloaden en tekst extraheren
def extract_text_from_pdf_url(pdf_url: str) -> str | None:
    """
    Downloadt een PDF via pdf_url en geeft de samengevoegde tekst terug.
    Retourneert None als het niet lukt.
    """
    if not pdf_url:
        return None

    try:
        r = requests.get(pdf_url, timeout=60)
        r.raise_for_status()
    except Exception as e:
        print(f"[WARN] pdf niet te downloaden: {pdf_url} ({e})")
        return None

    try:
        with pdfplumber.open(BytesIO(r.content)) as pdf:
            texts = []
            for page in pdf.pages:
                texts.append(page.extract_text() or "")
        full_text = "\n".join(texts).strip()
        return full_text if full_text else None
    except Exception as e:
        print(f"[WARN] kon tekst niet extraheren uit pdf: {pdf_url} ({e})")
        return None

In [52]:

PDF_HREF_PATTERNS = [
    ".pdf",
    "/file",
    "/pdf",
    "/binaries/",
    "open.overheid.nl/documenten/",
]

def _pdf_links_in_soup(soup, base_url: str) -> list[str]:
    """
    Vind alle links in deze soup die 'pdf-achtig' zijn.
    """
    links = []
    for a in soup.find_all("a"):
        href = a.get("href")
        if not href:
            continue
        href_abs = urljoin(base_url, href)
        text = (a.get_text() or "").lower()

        looks_like_pdf = any(p in href_abs for p in PDF_HREF_PATTERNS) or "pdf" in text
        if looks_like_pdf:
            links.append(href_abs)

    # dedup met volgorde-behoud
    seen = set()
    out = []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def find_pdf_urls(canonical_url: str) -> list[str]:
    """
    1) Zoek pdf-achtige links op de gegeven pagina.
    2) Zoek naar subdocument-links (/documenten/...) en volg die,
       om op die detailpagina's PDF-links te vinden.
    Retourneert ALLE gevonden PDF-URLs (uniek).
    """
    if not canonical_url:
        return []

    try:
        resp = requests.get(canonical_url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] pagina niet bereikbaar: {canonical_url} ({e})")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")

    pdf_urls = []

    # --- Stap 1: direct PDF-achtige links op deze pagina ---
    pdf_urls.extend(_pdf_links_in_soup(soup, canonical_url))

    # --- Stap 2: subdocument-links volgen (/documenten/...) ---
    subpages = []
    for a in soup.select('a[href^="/documenten/"]'):
        href = a.get("href")
        if not href:
            continue
        sub_url = urljoin(canonical_url, href)
        subpages.append(sub_url)

    # dedup subpages, en voorkom dat we de canonical zelf nogmaals doen
    seen_sub = set()
    subpages_unique = []
    for u in subpages:
        if u not in seen_sub and u != canonical_url:
            seen_sub.add(u)
            subpages_unique.append(u)

    # nu elke subpagina ophalen en PDF-links zoeken
    for sub_url in subpages_unique:
        try:
            r2 = requests.get(sub_url, timeout=30)
            r2.raise_for_status()
        except Exception as e:
            print(f"[WARN] subpagina niet bereikbaar: {sub_url} ({e})")
            continue

        soup2 = BeautifulSoup(r2.text, "html.parser")
        pdf_urls.extend(_pdf_links_in_soup(soup2, sub_url))

    # eind-deduplicatie
    seen = set()
    final = []
    for u in pdf_urls:
        if u not in seen:
            seen.add(u)
            final.append(u)

    if not final:
        print(f"[INFO] geen pdf-urls gevonden voor: {canonical_url}")

    return final


In [9]:
import re

keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|mistral|perplexity|ollama|LLaMA|openai|anthropic|midjourney|hugging face|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone|grok|xai|deepmind|azure"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X"
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}

_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='text'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    # print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='text'):

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


In [53]:
import ast 

# --- Functions ---
def split_into_sentences(text):
    if not isinstance(text, str):
        return []
    return re.split(r'(?<=[.!?])[\s\n]+', text)

def count_keywords(sentence, keywords):
    if not keywords:
        return 0
    return sum(1 for keyword in keywords if re.search(fr'\b{re.escape(keyword)}\b', sentence, re.I))

def merge_spans(spans):
    """Merge overlapping or adjacent spans into clusters."""
    if not spans:
        return []
    spans.sort()
    merged = [spans[0]]
    for s, e in spans[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:  # overlap or adjacency
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged

def extract_relevant_sections(row, text_col='text', keywords_col='matched_keywords_all', context_window=2):
    """
    Extract multiple keyword clusters with +/- context_window sentences.
    Returns blocks separated by blank lines.
    """
    text = row[text_col]
    keywords = row[keywords_col]

    if not isinstance(text, str):
        return ""
    if keywords is None or isinstance(keywords, float):
        keywords = []
    if isinstance(keywords, str):
        keywords = [keywords]

    sentences = split_into_sentences(text)
    if not sentences or not keywords:
        return ""

    relevant_indices = [i for i, s in enumerate(sentences) if count_keywords(s, keywords) > 0]
    if not relevant_indices:
        return ""

    # Build spans around each relevant index
    spans = []
    for i in relevant_indices:
        start = max(0, i - context_window)
        end = min(len(sentences), i + context_window + 1)
        spans.append((start, end))

    # Merge overlapping spans into clusters
    merged_spans = merge_spans(spans)

    # Collect blocks
    blocks = [" ".join(sentences[s:e]) for s, e in merged_spans]
    return "\n\n".join(blocks)

def prepare_for_rel(df):
    df['matched_keywords_all'] = df['matched_keywords_all'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    

    df['company_hits'] = df['company_hits'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    df['matched_keywords_all'] = df.apply(
    lambda row: (row['company_hits'] or []) + (row['matched_keywords_all'] or []),
    axis=1
    )
    return df

In [40]:
# test_df = pd.read_csv('C:\\Users\\joly-\\Github\\HUMAN\\subtitles\\subtitles_ai_related.csv')
# test_df_rel = prepare_for_rel(test_df)
# test_df_rel.head()

In [54]:
test_canonical = df['canonical'].iloc[0]
pdf_url = find_pdf_url(test_canonical)
text = extract_text_from_pdf_url(pdf_url) if pdf_url else None

print(f"PDF URL: {pdf_url}")
print(f"Extracted text (first 500 chars): {text if text else 'None'}")

PDF URL: https://open.overheid.nl/documenten/519e9e06-d4dd-451e-b2d2-ca6067f81b36/file
Extracted text (first 500 chars): Cee Sl he
AO:
MinisterievanFinanciën
TERBESLISSING
Aan
de staatssecretaris van Financiën — Fiscaliteit, Belastingdienst en Douane_
Directoraat-Generaal
voorFiscaleZaken
Directie Directe
Belastingen &Toeslagen
Persoonsgegevens
Nota naar aanleiding van het nader verslag en nota van
nota wijziging inzake het wetsvoorstel Wet onverplichte
tegemoetkoming onterechte afwijzing buitengerechtelijke
schuldregeling Datum
24 maart2025
Notanummer
2025-0000086426
Aanleiding Bijlagen
Het wetsvoorstel Wet onverplichte tegemoetkoming onterechte afwijzing I. Notanaaraanleidingvan
buitengerechtelijke schuldregeling (hierna: het wetsvoorstel) is op 18 december hetverslag
IL.Aanbiedingsbriefnotanaar
2024 bij de Tweede Kamer (TK) ingediend. Op 5 februari 2025 heeft deTK nader aanleidingvanhetverslag
verslag uitgebracht, Met deze nota wordt u gevraagd in stemmen met de nota TIL. Notavanwij

In [55]:
#specify which part to process
# part = 44
# subset = pd.read_csv(f"beleidsnotas_subset_part_{part}.csv")


df = pd.read_csv("beleidsnotas_rijksoverheid.csv")

final = pd.DataFrame()

for canonical in tqdm(df["canonical"], desc="PDF-tekst ophalen"):
    pdf_url = find_pdf_url(canonical)
    text = extract_text_from_pdf_url(pdf_url) if pdf_url else None
    title = df[df["canonical"] == canonical]["title"].iloc[0]

    # if text, apply AI-classification to decide whether to keep the text or not
    if text:
        temp_df = pd.DataFrame({"title": [title], "text": [text]})
        classified = ai_classification(temp_df, title_col="title", body_col="text")
        if classified['ai_related'].iloc[0] == "yes":
            print(f"[INFO] AI-gerelateerd document gevonden: {title} ({canonical})")
            classified = prepare_for_rel(classified)
            rel_text = extract_relevant_sections(classified.iloc[0], text_col='text', keywords_col='matched_keywords_all')
            # add classified info and relevant text to final dataframe
            row = df[df["canonical"] == canonical].iloc[0].to_dict()
            row["ai_related"] = classified['ai_related'].iloc[0]
            row["company_hits"] = classified['company_hits'].iloc[0]
            row['pdf_text'] = text
            row['relevant_text'] = rel_text
            row['matched_keywords'] = classified['matched_keywords_all'].iloc[0]
            row["type"] = "beleidsnota"
            final = pd.concat([final, pd.DataFrame([row])], ignore_index=True)
            
final.info()
final.to_csv("beleidsnotas_rijksoverheid_ai_related.csv", index=False)

PDF-tekst ophalen:   0%|          | 5/23743 [00:09<11:22:48,  1.73s/it]

[WARN] geen pdf-link gevonden op: https://www.rijksoverheid.nl/documenten/beleidsnota-s/2013/11/12/handhavingbeleidsplannen-bouwregelgeving


PDF-tekst ophalen:   0%|          | 9/23743 [00:47<62:17:21,  9.45s/it]

[INFO] AI-gerelateerd document gevonden: Investeren in Perspectief (Beleidsnota 2018) (https://www.rijksoverheid.nl/documenten/beleidsnota-s/2018/05/18/pdf-beleidsnota-investeren-in-perspectie)


PDF-tekst ophalen:   0%|          | 11/23743 [00:55<44:51:53,  6.81s/it]

...

[INFO] AI-gerelateerd document gevonden: Beslisnota bij Kamerbrief over tijdelijk beleidskader voor de bestrijding van drones (https://www.rijksoverheid.nl/documenten/beleidsnotas/2026/03/16/tk-beslisnota-bij-kamerbrief-tijdelijk-beleidskader-voor-de-bestrijding-van-drones)


PDF-tekst ophalen: 100%|█████████▉| 23713/23743 [10:50:05<00:31,  1.06s/it]

[WARN] geen pdf-link gevonden op: https://www.rijksoverheid.nl/documenten/beleidsnotas/2026/03/18/ek-beslisnota-bij-verslag-jbz-raad-5-en-6-maart-2026


PDF-tekst ophalen: 100%|██████████| 23743/23743 [10:50:44<00:00,  1.64s/it]


<class 'pandas.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                377 non-null    str   
 1   title             377 non-null    str   
 2   introduction      376 non-null    object
 3   canonical         377 non-null    str   
 4   dataurl           377 non-null    str   
 5   frontenddate      377 non-null    str   
 6   lastmodified      377 non-null    str   
 7   available         377 non-null    str   
 8   ai_related        377 non-null    str   
 9   company_hits      377 non-null    object
 10  pdf_text          377 non-null    str   
 11  relevant_text     377 non-null    str   
 12  matched_keywords  377 non-null    object
 13  type              377 non-null    str   
dtypes: object(3), str(11)
memory usage: 41.4+ KB


In [57]:
final.info()

<class 'pandas.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                377 non-null    str   
 1   title             377 non-null    str   
 2   introduction      376 non-null    object
 3   canonical         377 non-null    str   
 4   dataurl           377 non-null    str   
 5   frontenddate      377 non-null    str   
 6   lastmodified      377 non-null    str   
 7   available         377 non-null    str   
 8   ai_related        377 non-null    str   
 9   company_hits      377 non-null    object
 10  pdf_text          377 non-null    str   
 11  relevant_text     377 non-null    str   
 12  matched_keywords  377 non-null    object
 13  type              377 non-null    str   
dtypes: object(3), str(11)
memory usage: 41.4+ KB


In [43]:
# # delete Nan rows in pdf_text column
# df = df.dropna(subset=["pdf_text"])
# #how many rows dropped
# rows_dropped = len(pd.read_csv(f"beleidsnotas_subset_part_{part}.csv")) - len(df)
# print(f"Rijen verwijderd: {rows_dropped}")
# #save to new csv
# df.to_csv(f"beleidsnotas_part_{part}_with_pdf_text.csv", index=False)